# Week 8 lab · From press releases to word counts

**CM3801 · CMT309/509 · CMT124** · two-hour lab

**The question:** what do the Environment Agency's press releases about floods and about drought actually talk about, and how differently? To answer it you build one pipeline, step by step:

```
raw text → inspect → regex: extract & clean → tokens → normalised tokens → lemmas → counts by theme → interpretation
```

**You need:** today's two lecture hours (`W08_delivery.ipynb`): regex in Hour 1, NLTK in Hour 2. pandas `.str` methods (Week 6 and today's Hour 2), `groupby` (Week 6) and a bar chart (Week 7).

**Data:** `ea_flood_drought_press_2026.csv`: 34 press releases on **flooding** and **drought / dry weather** published by the Environment Agency on GOV.UK between January and September 2026, one row per release (`doc_id, published, title, summary, url, text`). Contains public sector information licensed under the Open Government Licence v3.0. The text is a plain-text export of each web page, so it is messy on purpose.

| Part | What you do |
|---|---|
| 0 | Understand the data |
| 1 | Regex: extract and clean |
| 2 | Tokenise and normalise |
| 3 | Stop words and lemmas |
| 4 | Analyse: themes, frequencies (POS and phrases *if time*) |
| 5 | Interpret |
| ★ | Optional challenge |

**How this lab works**
* Work through the parts in order: later parts use your earlier results. Each task has a goal, sometimes a hint (click to open) and an empty cell for your code.
* Run the **Checkpoint** cell after your code. If it prints `✔`, move on. If it raises an `AssertionError`, read the message, fix your code and try again.
* After Task T1.3 there is a cell that recreates the cleaned text, so that a problem there does not block the rest of the lab.
* **Your answer** cells ask you to explain a result in a few sentences (double-click to edit).

## Setup

Run both cells first. The second one downloads the NLTK data packages (once per runtime) and loads the press releases.

In [ ]:
# --- Setup: run this cell first -------------------------------------------------
# Course datasets live in the course repository under 2026-27/datasets/.
# get_data() uses a local copy if the notebook runs inside a copy of the repository;
# otherwise (e.g. in Google Colab) it downloads the file once and reuses it.
import os
import urllib.request

COURSE_REPO = "OWNER/REPO"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/main/2026-27/"


def course_file(relpath):
    """Return a local path to a course file such as 'datasets/owid_co2_cleaned.csv'."""
    for root in ("../../..", "."):                     # 1) running inside a copy of the repository
        path = os.path.join(root, relpath)
        if os.path.exists(path):
            return path
    path = os.path.join("course_files", relpath)       # 2) Colab: download once
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + relpath, path)
    return path


def get_data(name):
    """Return a local path to a course dataset, e.g. get_data('owid_co2_cleaned.csv')."""
    return course_file("datasets/" + name)

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# NLTK data packages (code and data are separate; once per runtime)
for name in ["punkt_tab", "stopwords", "wordnet", "averaged_perceptron_tagger_eng"]:
    print(f"{name:32s}", "ok" if nltk.download(name, quiet=True) else "FAILED")

press = pd.read_csv(get_data("ea_flood_drought_press_2026.csv"))
press.head()

---
## Part 0 · Understand the data

### T0.1 Look before you clean
Print the first 600 characters of the `text` of release `EA-026` twice: with `print()` and with `repr()`. `repr()` shows the characters that `print()` hides.

In [ ]:
# STUDENT TASK
# Task: print the first 600 characters of EA-026 with print() and with repr()
# your code here


### T0.2 What will get in the way?
Scroll through two or three releases (try `EA-001`, `EA-009`, `EA-028`). List **at least four** things in the text that will get in the way of counting words.

✍️ *Your answer:* 

### T0.3 How big is the corpus?
Add a column `n_words` (words separated by whitespace, `str.split()`), then show its summary statistics and the title of the longest release. Count releases per month of publication too (the first 7 characters of `published` are `YYYY-MM`).

In [ ]:
# STUDENT TASK
# Task: n_words per release, its summary, the longest release; releases per month
# your code here


---
## Part 1 · Regex: extract and clean

### T1.1 Contact details
Find every **e-mail address** and every **phone number** in the corpus, and count how often each one appears. The phone numbers look like `0345 988 1188` or `0800 80 70 60`.

<details><summary>Hint</summary>

Join all texts into one string first: `corpus = "\n".join(press["text"])`. A phone number is a `0`, three more digits, then groups of digits separated by spaces: `\s` also matches the non-breaking space. `pd.Series(list_of_matches).value_counts()` counts them.
</details>

In [ ]:
# STUDENT TASK
# Task: all e-mail addresses and phone numbers, with counts
# your code here


In [ ]:
# Checkpoint T1.1
assert "0800 80 70 60" in phones, "the Environment Agency incident hotline is missing"
assert all("@" in e and " " not in e for e in emails)
print("✔ T1.1:", len(set(emails)), "e-mail addresses,", len(set(phones)), "phone numbers")

### T1.2 Money
Press releases love money. Extract every amount in pounds **with its unit**, e.g. `£1.4bn`, `£50 million`, `£13m`, `£250,000`. Use **two groups**: the number and the (optional) unit. Then convert each amount to **£ million** and find the five largest amounts and the releases they come from.

<details><summary>Hint</summary>

Pattern idea: `£`, then the number `(\d[\d,]*(?:\.\d+)?)`, then optional whitespace `\s?`, then an optional unit group `(bn|billion|m|million|k)?` and a word boundary. Look at the raw matches before converting: the text writes `£10.5billion`, `£65million` and `£10 billion` (with a non-breaking space). For the conversion, remove the commas, then multiply by 1000 for billions, by 1 for millions, by 0.001 for thousands, and divide by 1,000,000 when there is no unit.
</details>

In [ ]:
# STUDENT TASK
# Task: extract (number, unit) pairs for every £ amount, convert to £ million, show the five largest
# your code here


In [ ]:
# Checkpoint T1.2
assert len(money) >= 70, "too few amounts: does your pattern accept '£10.5billion' and '£65million'?"
assert money["gbp_million"].max() == 104000, "the largest amount should be £104 billion"
print("✔ T1.2")

**Look at the result:** the same £10.5 billion appears in many releases. What does that tell you about press releases as a data source? (Keep it in mind for Part 5.)

### T1.3 Write `clean_text()`
Write a function `clean_text(text)` that returns the text ready for tokenising. It must:
1. replace each Markdown link `[link text](url)` by its **link text** only;
2. remove any remaining URL (`http...` or `www...`) and every e-mail address;
3. remove the line markers `> `, `- ` and `## ` at the **start of a line**;
4. replace curly apostrophes and quotes (`’ ‘ “ ”`) by straight ones (`'` and `"`);
5. replace each run of whitespace (spaces, `\xa0`, newlines) by a single space, and strip both ends.

Test it with the **Checkpoint** cell, then apply it to every release: `press["clean"] = press["text"].apply(clean_text)`.

<details><summary>Hint</summary>

* Link: `\[([^\]]*)\]\([^)]*\)` → `r"\1"` (group 1 is the link text).
* Line markers need `flags=re.MULTILINE` so that `^` means "start of each line": `^(?:> |- |## )`.
* Order matters: replace links **before** removing URLs, and remove line markers **before** you collapse the newlines.
</details>

In [ ]:
# STUDENT TASK
# Task: clean_text(text) -> str, following steps 1-5
# your code here


In [ ]:
# Checkpoint T1.3
assert clean_text("See [the flood map](https://check-flooding.example/a) now.") == "See the flood map now."
assert clean_text("> We’re ready.\xa0\xa0\n\n- Item two") == "We're ready. Item two"
assert clean_text("Call 0800 80 70 60 or email floods@example.org.uk today") == "Call 0800 80 70 60 or email today"
assert clean_text("Visit https://www.gov.uk/check-flooding now") == "Visit now"
assert "clean" in press and press["clean"].str.contains("\xa0|\\]\\(|’").sum() == 0
print("✔ T1.3")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
# It defines a reference clean_text() (steps 1-5) and recreates press["clean"].
def clean_text(text):
    text = re.sub(r"\[([^\]]*)\]\([^)]*\)", r"\1", text)                 # 1. [text](url) -> text
    text = re.sub(r"(?:https?://|www\.)\S+", "", text)                     # 2. bare URLs
    text = re.sub(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+", "", text)               #    e-mail addresses
    text = re.sub(r"^\s*(?:> |- |## )", "", text, flags=re.MULTILINE)      # 3. line markers
    text = re.sub(r"[‘’]", "'", text)                                      # 4. curly -> straight
    text = re.sub(r"[“”]", '"', text)
    text = re.sub(r"\s+", " ", text)                                       # 5. whitespace (\s includes \xa0)
    return text.strip()

press["clean"] = press["text"].apply(clean_text)
print("press['clean'] ready:", len(press), "releases")

---
## Part 2 · Tokenise and normalise

### T2.1 Tokens
Tokenise every cleaned release with `nltk.word_tokenize` into a new column `tokens` (a list per row). How many tokens are there in total, and how many **distinct** tokens?

In [ ]:
# STUDENT TASK
# Task: press["tokens"] = list of tokens per release; total and distinct counts
# your code here


### T2.2 Normalise
Make a column `words`: the tokens in **lower case**, keeping only **alphabetic** tokens (`str.isalpha()`). How many distinct words are left? Name two kinds of information this step threw away.

In [ ]:
# STUDENT TASK
# Task: press["words"] = lower-case alphabetic tokens; distinct count
# your code here


In [ ]:
# Checkpoint T2.2
assert all(w.isalpha() and w == w.lower() for w in press.loc[0, "words"])
print("✔ T2.2")

✍️ *Your answer:* 

---
## Part 3 · Stop words and lemmas

### T3.1 Stop words, and a decision of your own
Remove the NLTK English stop words from `words` and show the **20 most frequent** words in the whole corpus (`nltk.FreqDist`). Some frequent words are not stop words in general but say nothing about *this* corpus. Choose **at most five** extra stop words, add them to a set `EXTRA_STOP`, and justify each one in a sentence.

In [ ]:
# STUDENT TASK
# Task: FreqDist of non-stop words over the corpus (top 20); then define EXTRA_STOP (at most 5 words)
# your code here


✍️ *Your answer:* 

### T3.2 A `preprocess()` function with POS-aware lemmas
Write `preprocess(text)` that returns a list of lemmas: tokenise → POS-tag → keep alphabetic tokens → lower-case → drop stop words (`STOP | EXTRA_STOP`) → lemmatise with the POS from the tag (`J`→`"a"`, `V`→`"v"`, `R`→`"r"`, otherwise `"n"`). Apply it to the **clean** text into a column `lemmas`.

<details><summary>Hint</summary>

Today's Hour 2 (part-of-speech tagging) has `wordnet_pos(tag)`. Tag the tokens **before** lower-casing: capital letters help the tagger. Drop stop words before *and* after lemmatising (e.g. `includes` is not a stop word, `include` is in your `EXTRA_STOP`).
</details>

In [ ]:
# STUDENT TASK
# Task: preprocess(text) -> list of lemmas; press["lemmas"]
# your code here


In [ ]:
# Checkpoint T3.2
assert preprocess("Rivers were running low.") == ["river", "run", "low"]
assert "lemmas" in press and press["lemmas"].str.len().min() > 50
print("✔ T3.2")

---
## Part 4 · Analyse

### T4.1 Label each release: flood or drought?
There is no theme column: make one with regex. Search `title + " " + summary` for flood words and for drought words, then cross-tabulate the two flags (`pd.crosstab`). Look at every release that is in **both** groups or in **neither**, refine your patterns, and decide the remaining cases by reading them. Store the result in `press["theme"]` (`"flood"` or `"drought"`).

<details><summary>Hint</summary>

Start with `flood|rain` and `drought|dry|heatwave` and look at the crosstab. Then read the summaries of the releases in both groups: a drought summary that says *"lack of rainfall"* contains `rain`. Keywords do not understand context. Use `\b` so that `dry` does not match inside other words.
</details>

In [ ]:
# STUDENT TASK
# Task: press["theme"] = "flood" / "drought", built from regex flags on title + summary
# your code here


In [ ]:
# Checkpoint T4.1
assert set(press["theme"]) == {"flood", "drought"}
print("✔ T4.1")
print(press.groupby("theme")["published"].agg(["count", "min", "max"]))

### T4.2 Most frequent lemmas per theme, as rates
The two groups differ in size, so compare **rates**: occurrences per 1,000 lemmas. Make a table with one row per lemma and the rate in each theme, then plot the **15 most frequent lemmas of each theme** side by side (two horizontal bar charts).

<details><summary>Hint</summary>

`lemmas_by_theme = press.groupby("theme")["lemmas"].sum()` concatenates the lists. For each theme, `nltk.FreqDist(...)` gives counts; divide by the number of lemmas and multiply by 1,000.
</details>

In [ ]:
# STUDENT TASK
# Task: rates per 1,000 lemmas for each theme; two bar charts of the top 15
# your code here


### T4.3 What makes each theme different?
Words like `water` are frequent in **both** themes. Find the lemmas that are **distinctive**: compute `rates["flood"] - rates["drought"]` and show the 10 lemmas most typical of each theme. Only use lemmas that occur at least 5 times in total, so that a single unusual release does not dominate.

In [ ]:
# STUDENT TASK
# Task: the 10 most distinctive lemmas of each theme (rate difference; total count >= 5)
# your code here


### T4.4 Part-of-speech profile *(if time)*
Do the two themes *write* differently, or only talk about different things? For each theme, tag all tokens of the clean texts (`nltk.pos_tag`) and compute the share of **nouns** (`NN…`), **verbs** (`VB…`), **adjectives** (`JJ…`), **adverbs** (`RB…`) and **numbers** (`CD`) among the tagged tokens. Show the shares as a table (rows: POS group; columns: theme).

In [ ]:
# STUDENT TASK
# Task: share of POS groups per theme (table)
# your code here


### T4.5 Two-word phrases *(if time)*
Single words split phrases like *Environment Agency* or *dry weather*. List the 10 most frequent **bigrams** of lemmas in each theme (`nltk.bigrams` on each release's `lemmas`, so that no bigram crosses from one release into the next).

In [ ]:
# STUDENT TASK
# Task: top 10 bigrams per theme (bigrams built inside each release)
# your code here


---
## Part 5 · Interpret
Answer in 2-4 sentences each, using **your** outputs.

**Q1.** Using T4.2 and T4.3 (and T4.4-T4.5 if you did them): how does the Environment Agency talk about floods compared with drought? Give at least three pieces of evidence (words, phrases or POS shares).

✍️ *Your answer:* 

**Q2.** Which preprocessing decision changed your results the most: cleaning, stop words, extra stop words, or lemmatisation? How do you know?

✍️ *Your answer:* 

**Q3.** Name two limitations of drawing conclusions from these word counts.

✍️ *Your answer:* 

---
## ★ Optional challenge: the "weather whiplash" timeline
Release EA-010 speaks of a *weather whiplash*. Can you see it in the text? For each month, count how often **flood words** (`flood`, `flooding`, `floods`, `flooded`) and **drought words** (`drought`, `dry`, `heatwave`) appear in that month's clean texts, **per 1,000 words**, and plot both as lines over January-September.

In [ ]:
# STUDENT TASK
# Task: monthly rate of flood words and drought words per 1,000 words; line plot
# your code here


## Wrap-up

You turned 34 messy web pages into an analysis: regex to extract and clean, NLTK to tokenise, tag and lemmatise, pandas to label, count and compare, and a written interpretation that says what the counts can and cannot show. Every choice you made (the cleaning steps, the extra stop words, the theme rule) changed the result: that is the main lesson of text processing.